# 02 · Discretization and the Green's matrix

Chapter 01 ended with a preview: once the fault geometry is fixed, the entire
forward calculation can be written as one matrix multiplication,
$\mathbf{d} = G\mathbf{m}$. This chapter opens that matrix up. We will see
where its columns come from, build the whole matrix by hand, and then let
GeoDef assemble it in a single call. Along the way we will meet the first
warning sign of the inverse problem: patches whose surface signatures are so
similar that no measurement could tell them apart.

**Learning objectives**

By the end of the chapter, you will be able to:

- explain how dividing a fault into patches turns a continuous slip field into
  a finite list of unknowns
- recognize the design matrix of a straight-line fit as a tiny version of $G$
- build the Green's matrix one column at a time and with `fault.greens_matrix`
- describe what one row and one column of $G$ each mean physically
- use GeoDef's disk cache to avoid recomputing an expensive matrix
- explain what the condition number says about a discretization

**Prerequisites:** Chapters 00 and 01; matrix multiplication and least squares
**Estimated time:** 60–90 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. From a continuous fault to a list of numbers

A real fault surface can slip by a different amount at every point, so
describing its slip exactly would take infinitely many numbers. A computer can
only solve for finitely many. **Discretization** is the step that bridges the
two: we divide the fault into patches and assume slip is constant within each
patch. The unknowns of the problem become one slip value per patch, collected
into the model vector $\mathbf{m}$.

In mathematical language, we are approximating the continuous slip field by a
sum of simple building blocks,

$$s(\xi,\eta)\approx\sum_{j=1}^{N}m_j\,\phi_j(\xi,\eta).$$

Here $\xi$ and $\eta$ are positions on the fault surface, $N$ is the number of
patches, and each **basis function** $\phi_j$ equals one on patch $j$ and zero
everywhere else. The coefficient $m_j$ is simply the slip on patch $j$. This is
the same idea as representing a photograph with pixels: within one pixel, the
color is constant.

Smaller patches can represent more detailed slip, just as smaller pixels can
represent a sharper image. It is tempting to conclude that finer is always
better, but refining the patches does not add any new measurements at the
surface. It only adds unknowns, and neighboring small patches produce nearly
identical surface signals. We will measure that effect at the end of this
chapter, and Chapter 03 shows the trouble it causes.

## 2. Warm-up: the smallest useful design matrix

Any linear model can be written $\mathbf{d} = G\mathbf{m}$, where each column
of $G$ describes how one model parameter contributes to the data. For this
reason $G$ is often called the **design matrix**. Before building a fault
matrix with hundreds of columns, let us build one with two.

Fitting a straight line $y = a x + b$ to scattered points is a linear problem.
Stacking every observation into a vector gives

$$ \mathbf{y} = G\mathbf{m},\qquad
G = \begin{bmatrix} x_1 & 1\\ x_2 & 1\\ \vdots & \vdots\\ x_M & 1\end{bmatrix},\qquad
\mathbf{m} = \begin{bmatrix} a\\ b\end{bmatrix}. $$

The first column holds the $x$ values; multiplying it by $a$ gives the sloped
part of every prediction. The second column is all ones; multiplying it by $b$
shifts every prediction by the intercept. Fault-slip modeling uses exactly the
same structure. Only the columns change: instead of `x` and `1`, they become
elastic displacement fields.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

First we manufacture some noisy synthetic data along a known line. The seeded
random generator (Chapter 00, Section 22) makes the noise reproducible: every
run of this notebook draws the same "random" values.

In [ ]:
rng = np.random.default_rng(7)
x = np.linspace(0, 10, 30)
y = 1.8 * x - 0.7 + rng.normal(0, 0.8, x.size)

Now build the two-column design matrix and solve the least-squares problem,
exactly as in Chapter 00, Section 21. `np.column_stack` places the two columns
side by side.

In [ ]:
G_line = np.column_stack([x, np.ones_like(x)])
line_parameters, *_ = np.linalg.lstsq(G_line, y, rcond=None)
print(f"recovered slope     = {line_parameters[0]:.2f}  (true value 1.8)")
print(f"recovered intercept = {line_parameters[1]:.2f}  (true value -0.7)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, s=18, label="data")
ax.plot(x, G_line @ line_parameters, color="tab:red", label="least-squares fit")
ax.set(xlabel="x", ylabel="y", title="A two-column design matrix at work")
ax.legend()
plt.show()

**Checkpoint.** In the line fit, what would a third column of all $x_i^2$
values represent?

<details><summary>Answer</summary>

A quadratic term. Its coefficient would scale a parabola-shaped contribution,
turning the model into $y = a x + b + c x^2$. Adding a column always means
adding one model parameter and its contribution pattern.

</details>

## 3. From lines to faults

Now replace the two columns with hundreds: one per fault-patch slip component.
Each column of the fault $G$ is the surface displacement field produced by
**one meter of slip on a single patch**, with every other patch held at zero.
Scaling each column by the actual slip on its patch and summing reproduces the
full deformation field. That sum is exactly the matrix product $G\mathbf{m}$.

We will use a small fault so the matrix stays easy to inspect: 8 patches along
strike and 4 down dip, giving 32 patches.

In [ ]:
fault = geodef.Fault.planar(
    lat=0.0, lon=100.0, depth=10e3,
    strike=315.0, dip=20.0,
    length=40e3, width=16e3,
    n_length=8, n_width=4,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
print(f"{N} patches ({n_length} along strike x {n_width} down dip)")

The observations will come from a 9 by 9 grid of GNSS stations above the
fault. Each station records three displacement components (East, North, Up),
and each patch can slip in two directions (strike slip and dip slip). That
fixes the size of $G$ before we compute a single entry: three rows per station
and two columns per patch.

In [ ]:
grid_lon, grid_lat = np.meshgrid(np.linspace(99.6, 100.4, 9),
                                 np.linspace(-0.4, 0.4, 9))
station_lon = grid_lon.ravel()
station_lat = grid_lat.ravel()
n_stations = station_lon.size
print(f"{n_stations} stations  ->  G has {3 * n_stations} rows and {2 * N} columns")

## 4. Build $G$ one column at a time

Column $k$ of $G$ is nothing more than the forward model of Chapter 01 run
with unit slip on patch $k$ only. We can therefore build the entire matrix
with a loop: switch on one meter of slip, predict the displacements, and store
them in the matching column.

Recall the layout conventions from Chapter 01: the first $N$ columns belong to
strike slip and the next $N$ columns to dip slip, while the rows interleave the
three components station by station, `[e1, n1, u1, e2, n2, u2, ...]`.

The next cell fills the strike-slip columns.

In [ ]:
G_manual = np.zeros((3 * n_stations, 2 * N))

for k in range(N):
    unit_slip = np.zeros(N)
    unit_slip[k] = 1.0                      # 1 m of strike slip on patch k only
    ue, un, uz = fault.displacement(station_lat, station_lon,
                                    slip_strike=unit_slip, slip_dip=0.0)
    G_manual[0::3, k] = ue                  # East rows
    G_manual[1::3, k] = un                  # North rows
    G_manual[2::3, k] = uz                  # Up rows

The dip-slip columns follow the same recipe. They land in the second column
block, at positions `N + k`.

In [ ]:
for k in range(N):
    unit_slip = np.zeros(N)
    unit_slip[k] = 1.0                      # 1 m of dip slip on patch k only
    ue, un, uz = fault.displacement(station_lat, station_lon,
                                    slip_strike=0.0, slip_dip=unit_slip)
    G_manual[0::3, N + k] = ue
    G_manual[1::3, N + k] = un
    G_manual[2::3, N + k] = uz

print("built G by hand:", G_manual.shape)

## 5. The same matrix in one call

Looping over patches works, but it repeats setup work on every pass and would
become slow for a large fault. `fault.greens_matrix` assembles the very same
matrix in one vectorized call, which is how you will normally build it.

In [ ]:
G = fault.greens_matrix(station_lat, station_lon)
print("matches the hand-built matrix:", np.allclose(G, G_manual))

When your observations live in a GeoDef dataset rather than in plain
coordinate arrays, `geodef.greens.matrix(fault, dataset)` builds the matrix
and then projects each column into whatever that dataset actually measures.
Chapter 05 uses this to handle radar data that records only one viewing
direction. A three-component GNSS dataset measures East, North, and Up
directly, so for the dataset below the projection changes nothing and the two
matrices agree exactly.

The `gnss` constructor requires measurement values and uncertainties, so we
fill them with placeholder zeros and ones; only the station coordinates matter
for building $G$.

In [ ]:
zeros = np.zeros(n_stations)
ones = np.ones(n_stations)
stations = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=zeros, north=zeros, up=zeros,
    sigma_east=ones, sigma_north=ones, sigma_up=ones,
)
G_dataset = geodef.greens.matrix(fault, stations)
print("matches greens_matrix:", np.allclose(G_dataset, G))

## 6. Sidebar: GeoDef caches $G$ for you

Every entry of $G$ comes from evaluating the Okada solution for one
patch-station pair, so assembling $G$ for a large fault and dense data can
take a while. The same matrix is usually needed many times: every trial
inversion, every regularization experiment. `geodef.greens.matrix` therefore
fingerprints its inputs (the fault geometry, the observation points, the
elastic medium, and the numerical precision) and saves the result to disk. The
first call computes the matrix; identical later calls simply load it.

To keep this demonstration self-contained, we point the cache at a throwaway
temporary directory first.

In [ ]:
import tempfile
import time

geodef.cache.set_dir(tempfile.mkdtemp())   # a fresh, empty cache directory

In [ ]:
start = time.perf_counter()
G_first = geodef.greens.matrix(fault, stations)    # computes, then saves
middle = time.perf_counter()
G_second = geodef.greens.matrix(fault, stations)   # loads from disk
end = time.perf_counter()

print(f"first call  (compute + save): {1e3 * (middle - start):6.1f} ms")
print(f"second call (load from disk): {1e3 * (end - middle):6.1f} ms")
print("identical result:", np.allclose(G_first, G_second))

The fingerprint includes everything that could change the answer. Move one
station, change the shear modulus, or refine the patches, and GeoDef computes
a fresh matrix instead of silently returning a stale one. You will rarely need
to touch the cache directly, but `geodef.cache.info()`, `clear()`, and
`set_dir()` are available when you do; see
[the cache documentation](../docs/cache.md).

## 7. Reading the structure of $G$

We can display the whole matrix as an image, with one small colored square per
entry. Two organizing patterns stand out:

- **Columns are blocked.** The left half, `G[:, :N]`, holds the strike-slip
  responses; the right half, `G[:, N:]`, holds the dip-slip responses. The
  vertical line marks the boundary.
- **Rows are interleaved by station.** Reading down one column, the entries
  cycle through East, North, and Up for station 1, then station 2, and so on.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
vmax = np.abs(G).max()
image = ax.imshow(G, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.axvline(N - 0.5, color="k", lw=1.2)
ax.set_xlabel("model parameter (column):  strike slip | dip slip")
ax.set_ylabel("observation (row):  [e, n, u] for each station")
ax.set_title("The Green's matrix G")
fig.colorbar(image, ax=ax, label="displacement per unit slip (m / m)")
plt.show()

Each entry is a displacement (in meters) caused by one meter of slip, so the
entries are dimensionless ratios. One **row** collects how every patch affects
a single measurement. One **column** collects how a single patch affects every
measurement; it is that patch's surface "footprint."

### One column is one patch's footprint

To make the footprint idea concrete, we pull a single dip-slip column back out
of $G$ and un-interleave its rows into East, North, and Up values. The helper
`fault.patch_index` converts a (strike, dip) grid position into the patch
number, saving us from guessing how patches are ordered.

In [ ]:
k = fault.patch_index(strike_idx=4, dip_idx=1)   # a patch near the fault center
column = G[:, N + k]                             # its dip-slip column
col_e, col_n, col_u = column[0::3], column[1::3], column[2::3]
print(f"patch {k}: peak horizontal response "
      f"{np.hypot(col_e, col_n).max() * 100:.1f} cm per meter of slip")

Plotting the column as arrows over the fault shows exactly what it is: the
surface motion caused by one meter of dip slip on the highlighted patch. The
placeholder uncertainties below are only there because the dataset constructor
requires them; the plot uses just the arrows.

In [ ]:
footprint = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=col_e, north=col_n, up=col_u,
    sigma_east=ones, sigma_north=ones, sigma_up=ones,
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
highlight = np.zeros(N)
highlight[k] = 1.0
geodef.plot.slip(fault, highlight, ax=ax, cmap="Reds", colorbar=False,
                 coords="geographic")
arrow_scale = 12.0 / np.hypot(col_e, col_n).max()
geodef.plot.vectors(footprint, fault, ax=ax, components="horizontal",
                    scale=arrow_scale,
                    title=f"Surface response to 1 m of dip slip on patch {k}")
plt.show()

## 8. Predicting data for any slip model

With $G$ assembled, predicting the data for any slip distribution is a single
matrix multiplication. Notice how little code the physics takes now: all of the
elastic theory from Chapter 01 is stored inside the matrix.

We first build a slip model with two separate concentrations of slip, often
called **asperities**. The two index arrays convert each patch number into its
along-strike and down-dip grid position, and each asperity is a small
bell-shaped bump like the one in Chapter 01.

In [ ]:
along = np.arange(N) % n_length     # along-strike position of each patch
down = np.arange(N) // n_length     # down-dip position of each patch
slip = (1.0 * np.exp(-(((along - 2) / 1.2) ** 2 + ((down - 1) / 1.0) ** 2))
        + 0.7 * np.exp(-(((along - 6) / 1.0) ** 2 + ((down - 2) / 0.8) ** 2)))

The model vector follows the blocked layout: the strike-slip half is all
zeros, and the dip-slip half holds our two asperities. One `@` then predicts
every component at every station.

In [ ]:
m = np.concatenate([np.zeros(N), slip])   # [strike-slip block | dip-slip block]
d = G @ m
pred_e, pred_n, pred_u = d[0::3], d[1::3], d[2::3]
prediction = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=pred_e, north=pred_n, up=pred_u,
    sigma_east=ones, sigma_north=ones, sigma_up=ones,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
geodef.plot.slip(fault, slip, ax=axes[0], cmap="YlOrRd", coords="geographic",
                 colorbar_label="Dip slip (m)",
                 title="Slip model with two asperities")
arrow_scale = 12.0 / np.hypot(pred_e, pred_n).max()
geodef.plot.vectors(prediction, fault, ax=axes[1], components="both",
                    scale=arrow_scale, title="Predicted displacements")
plt.show()

The two asperities blur together into one broad surface pattern. Keep that
observation in mind: if two different slip models can produce nearly the same
surface data, then surface data alone cannot fully distinguish them. That is
the central difficulty of the next two chapters.

## 9. How similar are the columns?

The line fit at the start of this chapter worked because its two columns were
clearly different: no multiple of the sloped column looks like the constant
column. Fault columns are not so obliging. Two adjacent deep patches sit far
from the stations, and their footprints are nearly identical smears.

A single number summarizes how distinguishable a matrix's columns are: the
**condition number**. It compares the matrix's strongest and weakest responses
to a model change. A condition number near one describes a comfortable,
well-behaved problem. A very large one warns that some combination of patches
barely affects the data at all, so measurement noise can masquerade as large
slip on those patches.

In [ ]:
print(f"condition number of G: {np.linalg.cond(G):.2e}")

Now refine the same fault to 12 by 6 patches, keeping the stations fixed, and
build the matrix again. The finer patches are individually weaker sources and
their footprints overlap more, so the condition number grows.

In [ ]:
finer_fault = geodef.Fault.planar(
    lat=0.0, lon=100.0, depth=10e3,
    strike=315.0, dip=20.0,
    length=40e3, width=16e3,
    n_length=12, n_width=6,
)
G_finer = finer_fault.greens_matrix(station_lat, station_lon)
print(f"patches: {N} -> {finer_fault.n_patches}")
print(f"condition number: {np.linalg.cond(G):.2e} -> {np.linalg.cond(G_finer):.2e}")

The condition number measures *numerical distinguishability*, not geological
realism. Refining the grid lets the model represent finer slip detail, but it
also manufactures unknowns the data cannot separate. Choosing a discretization
is therefore a modeling decision, not a mesh-generation chore. Chapter 03
shows what happens when this warning is ignored.

## Checkpoint questions

**What physical object is one column of $G$?**

<details><summary>Answer</summary>

The set of displacements measured at every observation point when one patch
slips by exactly one meter in one direction (strike slip or dip slip) and
every other patch stays locked. It is that patch's surface footprint.

</details>

**Does using smaller patches improve resolution?**

<details><summary>Answer</summary>

Not by itself. Smaller patches improve the model's ability to *represent*
detailed slip, but resolution improves only if the data can tell the new
columns apart. Refining the mesh adds unknowns without adding measurements.

</details>

**Why must the cache fingerprint include the elastic medium and precision, not
just the fault and stations?**

<details><summary>Answer</summary>

Changing either one changes the computed matrix values. If they were left out
of the fingerprint, a later call with a different shear modulus could silently
load a matrix computed with the old one, and every result downstream would use
the wrong physics.

</details>

## Common mistakes

- **Guessing the patch order.** Reshaping slip with `reshape(n_width,
  n_length)` when the order is actually the other way around silently
  transposes the fault. Use `fault.grid_shape`, `fault.patch_index`, and the
  reshape helpers instead of memorized index arithmetic.
- **Forgetting the column blocks.** The first $N$ columns are strike slip and
  the next $N$ are dip slip. Placing a dip-slip model in the first block
  produces confidently wrong predictions.
- **Comparing condition numbers across different problems.** The condition
  number depends on units and scaling as well as geometry. It is most useful
  for comparing variants of the *same* problem, such as two refinements of one
  fault.
- **Trusting a cache blindly.** GeoDef's cache keys include the full physics,
  but if you build your own caching for other quantities, an incomplete key
  will eventually hand you stale results.

## Recap

- Discretization replaces a continuous slip field with one constant slip value
  per patch; the patch values form the model vector $\mathbf{m}$.
- The Green's matrix is built column by column from unit-slip forward models;
  `fault.greens_matrix` and `geodef.greens.matrix` assemble and cache it.
- Columns are a patch's surface footprint; rows are a single measurement's
  sensitivity to every patch.
- Refining patches increases what the model can represent but makes columns
  more similar, which raises the condition number.

Chapter 03 tries to invert this matrix directly and meets the consequences of
those similar columns head-on.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/02_discretization_and_g_matrix_solutions.ipynb`.

1. **Refinement and conditioning.** Rebuild the fault at several refinements
   (for example 4 by 2, 8 by 4, and 12 by 6 patches) with the same stations.
   Tabulate the shape of $G$ and its condition number at each refinement.
2. **Two mechanisms, one patch.** For a single patch, extract the strike-slip
   column and the dip-slip column and plot each as a vector field. Describe how
   the two footprints differ.
3. **Shallow versus deep similarity.** Pick two adjacent shallow patches and
   two adjacent deep patches. For each pair, compute the normalized dot product
   of their dip-slip columns (the cosine of the angle between them). Which pair
   is harder to tell apart, and why?
4. **Challenge: a one-component instrument.** Build the rows of $G$ for an
   imaginary instrument that measures only the East component of displacement,
   and verify them against the East rows of the full three-component matrix.

## Further reading

- Aster, R., Borchers, B., and Thurber, C. (2018), *Parameter Estimation and
  Inverse Problems*, 3rd ed., Chapters 1–3 on discretized linear inverse
  problems.
- Menke, W. (2018), *Geophysical Data Analysis: Discrete Inverse Theory*,
  4th ed., on design matrices and conditioning.
- Okada, Y. (1985), the rectangular dislocation solution behind every column,
  introduced in Chapter 01.
- [GeoDef Green's function documentation](../docs/greens.md) and
  [cache documentation](../docs/cache.md).
- The next course chapter is `03_unregularized_inversion.ipynb`.